<a href="https://colab.research.google.com/github/ritaireriy-wz/AUS_with_Ireri/blob/main/NNEA_GPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import math
import re
import statistics as st
import sys
import os

def parse_nmea(path):
    recs = {'GGA': [], 'RMC': [], 'GSA': [], 'GSV': [], 'VTG': []}
    bad = 0
    total = 0
    with open(path, encoding='ascii', errors='replace') as f:
        for raw in f:
            line = raw.strip().rstrip('\\').strip()
            if not line.startswith('$'):
                continue
            total += 1
            body, _, cs = line[1:].partition('*')
            # checksum verification
            calc = 0
            for ch in body:
                calc ^= ord(ch)
            if cs and cs[:2].upper() != f'{calc:02X}':
                bad += 1
            fields = body.split(',')
            kind = fields[0][2:]
            if kind in recs:
                recs[kind].append(fields)
    return recs, total, bad

def dm_to_deg(dm, hemi):
    if not dm:
        return None
    dot = dm.index('.')
    deg = float(dm[:dot - 2])
    minutes = float(dm[dot - 2:])
    val = deg + minutes / 60.0
    return -val if hemi in ('S', 'W') else val

def to_f(x):
    try:
        return float(x)
    except ValueError:
        return None

def haversine(lat1, lon1, lat2, lon2):
    R = 6371008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = p2 - p1
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def time_s(t):
    h, m, s = int(t[0:2]), int(t[2:4]), float(t[4:])
    return h * 3600 + m * 60 + s

def analyse(path, label):
    recs, total, bad = parse_nmea(path)
    print(f'===== {label}: {path} =====')
    print(f'sentences: {total}, checksum failures: {bad}')
    print({k: len(v) for k, v in recs.items()})

    # GGA: time, lat, lon, fix quality, nsat, hdop, alt
    gga = []
    for f in recs['GGA']:
        t = f[1]
        lat = dm_to_deg(f[2], f[3]); lon = dm_to_deg(f[4], f[5])
        fixq = int(f[6]) if f[6] else 0
        nsat = int(f[7]) if f[7] else 0
        hdop = to_f(f[8]); alt = to_f(f[9]); geoid = to_f(f[11])
        gga.append(dict(t=t, ts=time_s(t), lat=lat, lon=lon, q=fixq, nsat=nsat,
                        hdop=hdop, alt=alt, geoid=geoid))
    fixed = [g for g in gga if g['q'] > 0 and g['lat'] is not None]
    print(f'GGA epochs: {len(gga)}, with fix: {len(fixed)}, no fix: {len(gga)-len(fixed)}')
    if gga:
        print(f'time span: {gga[0]["t"]} -> {gga[-1]["t"]}  '
              f'({gga[-1]["ts"]-gga[0]["ts"]:.0f} s)')
    if fixed:
        print(f'first fix: {fixed[0]["t"]}, last fix: {fixed[-1]["t"]}')

    # RMC: date, status, speed (knots), course
    rmc = []
    for f in recs['RMC']:
        rmc.append(dict(t=f[1], status=f[2], lat=dm_to_deg(f[3], f[4]),
                        lon=dm_to_deg(f[5], f[6]), sog=to_f(f[7]),
                        cog=to_f(f[8]), date=f[9]))
    if rmc:
        print(f'RMC status counts: A={sum(r["status"]=="A" for r in rmc)}, '
              f'V={sum(r["status"]=="V" for r in rmc)}, date={rmc[0]["date"]}')

    # GSA: PDOP, HDOP, VDOP, fix type
    gsa = []
    for f in recs['GSA']:
        if len(f) >= 18:
            used = [x for x in f[3:15] if x]
            gsa.append(dict(mode=f[1], fix=f[2], nused=len(used),
                            pdop=to_f(f[15]), hdop=to_f(f[16]), vdop=to_f(f[17].split("*")[0])))
    valid = [g for g in gsa if g['pdop'] is not None and g['pdop'] < 50]
    print(f'GSA: {len(gsa)} sentences, {len(valid)} with plausible DOP (<50), '
          f'{len(gsa)-len(valid)} with corrupt DOP (99.99 or similar)')
    if valid:
        print(f'  HDOP median {st.median(g["hdop"] for g in valid):.2f} '
              f'(min {min(g["hdop"] for g in valid):.2f}, max {max(g["hdop"] for g in valid):.2f})')
        print(f'  VDOP median {st.median(g["vdop"] for g in valid):.2f}')

    # GSV: satellites in view and SNR
    snrs = []
    prns_seen = set()
    for f in recs['GSV']:
        for i in range(4, len(f) - 3, 4):
            prn = f[i]; snr = f[i + 3].split('*')[0]
            if prn:
                prns_seen.add(prn)
                if snr:
                    snrs.append(int(snr))
    print(f'satellite PRNs seen: {sorted(prns_seen, key=int)}')
    if snrs:
        print(f'SNR (dB-Hz): median {st.median(snrs):.0f}, mean {st.mean(snrs):.1f}, '
              f'range {min(snrs)}-{max(snrs)}')

    # Coordinates in local metres relative to the fix centroid
    if len(fixed) < 2:
        return fixed, rmc, gsa
    lat0 = st.mean(g['lat'] for g in fixed)
    lon0 = st.mean(g['lon'] for g in fixed)
    mlat = 111132.954 - 559.822 * math.cos(2 * math.radians(lat0))
    mlon = 111132.954 * math.cos(math.radians(lat0))
    for g in fixed:
        g['e'] = (g['lon'] - lon0) * mlon
        g['n'] = (g['lat'] - lat0) * mlat
    return fixed, rmc, gsa

def stationary_stats(fixed):
    e = [g['e'] for g in fixed]; n = [g['n'] for g in fixed]
    print('\n--- Stationary-position statistics (from fixed epochs) ---')
    print(f'N = {len(fixed)} fixes')
    print(f'mean E {st.mean(e):.2f} m, mean N {st.mean(n):.2f} m')
    print(f'std E {st.stdev(e):.2f} m, std N {st.stdev(n):.2f} m')
    rad = [math.hypot(a - st.mean(e), b - st.mean(n)) for a, b in zip(e, n)]
    rad.sort()
    cep50 = rad[len(rad) // 2]
    r95 = rad[int(0.95 * (len(rad) - 1))]
    sig_h = math.sqrt(st.variance(e) + st.variance(n))
    print(f'radial error from mean: median (CEP50) {cep50:.2f} m, 95th percentile {r95:.2f} m, max {rad[-1]:.2f} m')
    print(f'2DRMS (2*sqrt(varE+varN)) = {2*sig_h:.2f} m;  R95 approx 2.45*sigma_h = {2.45*sig_h/ math.sqrt(1) * 1:.2f} m')
    alts = [g['alt'] for g in fixed if g['alt'] is not None]
    if alts:
        print(f'altitude MSL: mean {st.mean(alts):.2f} m, std {st.stdev(alts):.2f} m, range {min(alts):.1f}-{max(alts):.1f} m')
    hd = [g['hdop'] for g in fixed if g['hdop'] is not None]
    if hd:
        print(f'GGA HDOP median {st.median(hd):.2f}')

def moving_stats(fixed, rmc):
    print('\n--- Trajectory statistics ---')
    d = 0.0; seg_speeds = []
    for a, b in zip(fixed, fixed[1:]):
        dt = b['ts'] - a['ts']
        if dt <= 0:
            continue
        seg = haversine(a['lat'], a['lon'], b['lat'], b['lon'])
        d += seg
        seg_speeds.append(seg / dt)
    print(f'path length (sum of fix-to-fix distances): {d:.1f} m')
    e = [g['e'] for g in fixed]; n = [g['n'] for g in fixed]
    print(f'straight-line start->end: {math.hypot(fixed[-1]["e"]-fixed[0]["e"], fixed[-1]["n"]-fixed[0]["n"]):.1f} m')
    print(f'bounding box: E {min(e):.1f}..{max(e):.1f} m, N {min(n):.1f}..{max(n):.1f} m')
    print(f'max distance from start: {max(math.hypot(x-fixed[0]["e"], y-fixed[0]["n"]) for x,y in zip(e,n)):.1f} m')
    sog = [r['sog'] for r in rmc if r['sog'] is not None and r['status'] == 'A']
    if sog:
        print(f'RMC speed over ground (knots): median {st.median(sog):.2f}, mean {st.mean(sog):.2f}, max {max(sog):.2f}')
        print(f'   -> m/s: median {st.median(sog)*0.514444:.2f}, max {max(sog)*0.514444:.2f}')
    if seg_speeds:
        print(f'position-derived speed (m/s): median {st.median(seg_speeds):.2f}, max {max(seg_speeds):.2f}')
    # mean RMC speed vs. position-derived integral
    sog_ms = [r['sog']*0.514444 for r in rmc if r['sog'] is not None and r['status']=='A']
    if sog_ms:
        print(f'integral of RMC speed over fixed period (approx distance): {sum(sog_ms):.1f} m (1 Hz, summed)')
    # stationary periods
    stat = sum(1 for s in sog if s < 0.5)  # < 0.5 knot
    print(f'epochs with speed < 0.5 kn: {stat} of {len(sog)}')

# Run the main analysis loop using the absolute paths
for path, label in [('/content/nmea-test1-1.txt', 'FILE 1'), ('/content/nmea-test2-1.txt', 'FILE 2')]:
    if os.path.exists(path):
        fixed, rmc, gsa = analyse(path, label)
        if len(fixed) >= 2:
            stationary_stats(fixed)
            moving_stats(fixed, rmc)
        print()

===== FILE 1: /content/nmea-test1-1.txt =====
sentences: 1040, checksum failures: 0
{'GGA': 231, 'RMC': 231, 'GSA': 231, 'GSV': 116, 'VTG': 231}
GGA epochs: 231, with fix: 219, no fix: 12
time span: 090550.000 -> 090940.000  (230 s)
first fix: 090602.000, last fix: 090940.000
RMC status counts: A=219, V=12, date=041125
GSA: 231 sentences, 191 with plausible DOP (<50), 40 with corrupt DOP (99.99 or similar)
  HDOP median 2.17 (min 2.14, max 2.91)
  VDOP median 0.97
satellite PRNs seen: ['04', '05', '06', '07', '09', '16', '21', '27', '30', '33', '34', '35', '36', '37', '38', '39', '40', '41', '43', '44', '45', '46']
SNR (dB-Hz): median 29, mean 26.0, range 14-33

--- Stationary-position statistics (from fixed epochs) ---
N = 219 fixes
mean E 0.00 m, mean N 0.00 m
std E 4.08 m, std N 5.15 m
radial error from mean: median (CEP50) 5.59 m, 95th percentile 12.38 m, max 13.19 m
2DRMS (2*sqrt(varE+varN)) = 13.14 m;  R95 approx 2.45*sigma_h = 16.10 m
altitude MSL: mean 107.82 m, std 0.85 m, ran